# Dataset Self-Conformance Demo

Every bundled dataset should satisfy the same basic invariants:

1. Connectivity indices point to nodes that actually exist.
2. The declared basis is one `pyfieldml` recognises.
3. Evaluating the coordinate field at the reference-element centroid returns a finite point.

This notebook runs those checks over every dataset as a fail-fast sanity screen.

In [ ]:
import numpy as np

from pyfieldml import datasets
from pyfieldml.interop.meshio import _basis_topology_order, _find_basis_name

## Reference-element centroids

Each supported topology has its own centroid in xi-space:

In [ ]:
CENTROIDS = {
    "line": (0.5,),
    "triangle": (1.0 / 3.0, 1.0 / 3.0),
    "quad": (0.5, 0.5),
    "tet": (0.25, 0.25, 0.25),
    "hex": (0.5, 0.5, 0.5),
    "wedge": (1.0 / 3.0, 1.0 / 3.0, 0.5),
}

RECOGNIZED_TOPOLOGIES = set(CENTROIDS)

## Run the invariants over every bundled dataset

In [ ]:
def check(name):
    doc = datasets.load(name)
    coords = doc.evaluators["coordinates"].as_ndarray()
    conn = doc.evaluators["coordinates.connectivity"].as_ndarray().astype(np.int64)
    n_nodes = coords.shape[0]

    # (1) connectivity stays within node count (1-indexed)
    assert conn.min() >= 1, f"{name}: connectivity has index < 1"
    assert conn.max() <= n_nodes, (
        f"{name}: connectivity max {conn.max()} exceeds node count {n_nodes}"
    )

    # (2) basis is recognized
    basis = _find_basis_name(doc.region)
    topology, order = _basis_topology_order(basis)
    assert topology in RECOGNIZED_TOPOLOGIES, f"{name}: bad topology {topology!r}"

    # (3) evaluating at the element centroid returns finite numbers
    field = doc.field("coordinates")
    xi = CENTROIDS[topology]
    centroid = field.evaluate(element=1, xi=xi)
    assert np.all(np.isfinite(centroid)), f"{name}: non-finite centroid"
    return {
        "dataset": name,
        "n_nodes": n_nodes,
        "n_elems": conn.shape[0],
        "topology": topology,
        "order": order,
        "centroid": centroid,
    }


rows = [check(n) for n in datasets.list()]

## Summary table

In [ ]:
fmt = "{:22s} {:>7s} {:>7s} {:10s} {:>5s}  {}"
row_fmt = "{:22s} {:7d} {:7d} {:10s} {:5d}  {}"
header = fmt.format("dataset", "nodes", "elems", "topology", "order", "centroid")
print(header)
print("-" * len(header))
for row in rows:
    c = np.array2string(row["centroid"], precision=4, suppress_small=True)
    print(
        row_fmt.format(
            row["dataset"],
            row["n_nodes"],
            row["n_elems"],
            row["topology"],
            row["order"],
            c,
        )
    )
print()
print(f"all {len(rows)} bundled datasets pass self-conformance.")

### Why this matters

Bundled assets are the foundation for every downstream tutorial, benchmark, and user demo. This notebook is a lightweight smoke test you can run after any dataset edit to confirm nothing has silently corrupted. For a strict byte-level regression canary, see `tests/benchmarks/test_reproducibility.py`.